# DiffSpot: test any group model on a shared 44-pair sample

Use your own model's environment/kernel. The workflow is architecture/framework
independent: save sample ? run your existing predictor or plug in an adapter ?
score with your Design2Code threshold ? inspect boxes and per-case results.

**Preliminary smoke only.** Do not select/tune models, thresholds or prompts
using these results. Kimi/official judge evaluation is a separate workflow.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib

def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
from IPython.display import display, HTML

REPO = Path.cwd().resolve()
if not (REPO / 'scripts/diffspot_smoke.py').exists():
    REPO = REPO.parent
assert (REPO / 'scripts/diffspot_smoke.py').exists(), 'Open from repo root or notebooks/'
DATASET = Path(r'D:\csc409-data\diffspot-v1') if os.name == 'nt' else Path(os.environ.get('LAB_ROOT', '/virtual/.cache/' + os.environ.get('USER', '') + '/csc490')) / 'data/diffspot-v1'
SAMPLE = REPO / 'outputs/diffspot-smoke/sample44'
RUN = REPO / 'outputs/diffspot-smoke/my-model'
REVIEW = REPO / 'outputs/diffspot-smoke/my-review'
N, SEED = 44, 490

# Every member edits these for their OWN selected model.
ADAPTER = None  # e.g. 'my_model_adapter:create_predictor'; None = import predictions
MODEL_CONFIG = Path('/EDIT/my-selected-model.json')
PREDICTIONS = RUN / 'predictions.jsonl'  # or your existing inference output
VALIDATION_METRICS = Path('/EDIT/my-design2code-validation.metrics.json')
SUPPORTS_TYPE = False
FROM_FULL = False  # True only when importing all 4,400 predictions

def run(*args):
    subprocess.run([sys.executable, str(REPO/'scripts/diffspot_smoke.py'), *map(str,args)], cwd=REPO, check=True)
assert (DATASET/'manifest.json').exists(), 'Extract the prepared dataset ZIP outside the repo; set DATASET.'
print('Kernel:', sys.executable)
print('Dataset:', DATASET)


## 1. Create once, reuse across all models
Default: 13 operators ? 3 difficulties + 5 controls. Do not resample per model.

In [ ]:
if (SAMPLE/'sample.json').exists():
    print('Reusing saved sample:', SAMPLE)
else:
    run('sample','--dataset',DATASET,'--n',N,'--seed',SEED,'--out',SAMPLE)
selection = json.loads((SAMPLE/'sample.json').read_text(encoding='utf-8'))
print('Pairs:', selection['n'], '| no-diff:', selection['no_diff'], '| seed:', selection['seed'])


## 2. Run your model
**Import route:** leave ADAPTER=None and point PREDICTIONS at your existing output.
**Adapter route:** set ADAPTER and MODEL_CONFIG. See docs/diffspot-smoke.md for
the factory contract. No model is assumed or randomly initialized by this notebook.


In [ ]:
if ADAPTER is None:
    print('Import mode. Set PREDICTIONS to your model output:', PREDICTIONS)
else:
    assert MODEL_CONFIG.is_file(), 'Set the owner-defined model config JSON path.'
    if (RUN/'complete.json').exists():
        saved_run = json.loads((RUN/'run.json').read_text())
        assert saved_run['adapter'] == ADAPTER and saved_run['model_config_sha256'] == sha(MODEL_CONFIG), 'Run settings changed: choose a new RUN directory.'
        print('Reusing completed run:', RUN)
    else:
        run('predict','--dataset',DATASET,'--sample',SAMPLE,'--adapter',ADAPTER,'--model-config',MODEL_CONFIG,'--out',RUN)


## 3. Score with the existing Design2Code threshold
No threshold is selected on these samples. Official description scores are not binary detector scores.

In [ ]:
assert PREDICTIONS.is_file(), 'Supply your model predictions or run your adapter first.'
assert VALIDATION_METRICS.is_file(), 'Set your actual Design2Code validation metrics path.'
if (REVIEW/'metrics.json').exists():
    saved_review = json.loads((REVIEW/'metrics.json').read_text())
    assert saved_review['predictions_sha256'] == sha(PREDICTIONS), 'Predictions changed: choose a new REVIEW directory.'
    assert saved_review['validation_metrics_sha256'] == sha(VALIDATION_METRICS), 'Threshold source changed: choose a new REVIEW directory.'
    assert saved_review['sample_sha256'] == sha(SAMPLE/'sample.json'), 'Sample changed: choose a new REVIEW directory.'
    print('Reusing verified review:', REVIEW)
else:
    flags = (['--supports-type'] if SUPPORTS_TYPE else []) + (['--from-full'] if FROM_FULL else [])
    run('review','--dataset',DATASET,'--sample',SAMPLE,'--predictions',PREDICTIONS,'--validation-metrics',VALIDATION_METRICS,'--out',REVIEW,*flags)
metrics = json.loads((REVIEW/'metrics.json').read_text(encoding='utf-8'))
display(metrics)


## 4. Inspect localization and failure cases
Green = visible target, blue = prediction. Use this only after inference; never send this annotated review to a model.

In [ ]:
display(HTML((REVIEW/'review.html').read_text(encoding='utf-8')))
print('Portable review:', REVIEW/'review.html')
print('Per-case table:', REVIEW/'cases.csv')
